# Exploratory Data Analysis: CUBIT-Seg vs. DeepCrack

This notebook analyzes the structural properties and domain differences between:
- **Source Domain (`DeepCrack`)**: 537 images (road/concrete close-ups)
- **Target Evaluation Domain (`CUBIT-Seg`)**: 5,462 images (UAV/DSLR building facades)

Key objectives:
1. Verify image resolutions, empty-mask ratios, and dataset splits.
2. Quantify crack pixel ratios (class imbalance) across both datasets.
3. Compare approximate crack width distributions via Euclidean Distance Transform.
4. Visualize representative samples and save summary figures to `assets/`.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm

from src.data import CrackDataset, load_binary_mask
from src.paths import ASSETS_DIR, DATA_DIR

ASSETS_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 데이터 불러오기 (CUBIT-Seg, DeepCrack)

cubit_ds = CrackDataset(DATA_DIR / "cubit_seg", dataset_type="cubit_seg")
dc_train = CrackDataset(DATA_DIR / "deepcrack", dataset_type="deepcrack", split="train")
dc_test = CrackDataset(DATA_DIR / "deepcrack", dataset_type="deepcrack", split="test")

print(f"CUBIT-Seg total : {len(cubit_ds)}")
print(f"DeepCrack total : {len(dc_train) + len(dc_test)} (train={len(dc_train)}, test={len(dc_test)})")

In [ ]:
# 마스크 스캔 -> 균열 면적 비율 계산 

def collect_stats(dataset: CrackDataset, name: str) -> pd.DataFrame:
    records = []
    for pair in tqdm(dataset.pairs, desc=f"Scanning {name}"):
        mask = load_binary_mask(pair.mask_path)
        h, w = mask.shape
        crack_pixels = int(mask.sum())  # 균열 픽셀 개수 
        crack_ratio = crack_pixels / (h * w) * 100.0  # 균열 면적 비율 (%)

        if crack_pixels > 0:
            dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)  # 1인 픽셀에서 가장 가까운 0인 픽셀까지 거리 계산 (폭 구하기)
            # Approximate width (2 * radius) over foreground pixels > 0
            max_width = float(dist.max() * 2.0)
            mean_width = float(dist[mask > 0].mean() * 2.0)
        else:
            max_width = 0.0
            mean_width = 0.0

        records.append(
            {
                "dataset": name,
                "subset": pair.subset,
                "sample_id": pair.sample_id,
                "height": h,
                "width": w,
                "crack_pixels": crack_pixels,
                "crack_ratio_pct": crack_ratio,
                "is_empty": crack_pixels == 0,
                "approx_max_width_px": max_width,
                "approx_mean_width_px": mean_width,
            }
        )
    return pd.DataFrame(records)


df_cubit = collect_stats(cubit_ds, "CUBIT-Seg")
df_dc_train = collect_stats(dc_train, "DeepCrack")
df_dc_test = collect_stats(dc_test, "DeepCrack")
df_all = pd.concat([df_cubit, df_dc_train, df_dc_test], ignore_index=True)

summary = (
    df_all.groupby("dataset")
    .agg(
        num_images=("sample_id", "count"),
        empty_masks=("is_empty", "sum"),
        mean_crack_ratio_pct=("crack_ratio_pct", "mean"),
        median_crack_ratio_pct=("crack_ratio_pct", "median"),
        mean_width_px=("approx_mean_width_px", lambda s: s[s > 0].mean()),
        max_width_p95_px=("approx_max_width_px", lambda s: np.percentile(s[s > 0], 95)),
    )
    .round(3)
)
summary

In [ ]:
# 분포 히스토그램 그리기 (eda_distribution.png)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for name, color in [("CUBIT-Seg", "#2563eb"), ("DeepCrack", "#ea580c")]:
    sub = df_all[(df_all["dataset"] == name) & (~df_all["is_empty"])]
    axes[0].hist(
        sub["crack_ratio_pct"],
        bins=40,
        range=(0, 15),
        density=True,
        alpha=0.55,
        label=f"{name} (n={len(sub)})",
        color=color,
    )
    axes[1].hist(
        sub["approx_max_width_px"],
        bins=40,
        range=(0, 40),
        density=True,
        alpha=0.55,
        label=f"{name} (median={sub['approx_max_width_px'].median():.1f}px)",
        color=color,
    )

axes[0].set_title("Crack Pixel Ratio Distribution (%)")
axes[0].set_xlabel("Crack Area Ratio (%)")
axes[0].set_ylabel("Density")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].set_title("Approximate Max Crack Width Distribution (px)")
axes[1].set_xlabel("Approx. Max Width (px)")
axes[1].set_ylabel("Density")
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()
dist_path = ASSETS_DIR / "eda_distributions.png"
plt.savefig(dist_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {dist_path}")

In [ ]:
# 대표 샘플 시각화 (eda_samples.png)
# 각 데이터셋에서 면적 하위 30%와 상위 80%의 이미지 뽑아서 저장)

def pick_representative_indices(df: pd.DataFrame, k: int = 2) -> list[int]:
    non_empty = df[~df["is_empty"]].sort_values("crack_ratio_pct")
    if len(non_empty) < k:
        return list(range(min(k, len(df))))
    # Pick around 30th and 80th percentiles to show thin and thick cracks
    pos = [int(len(non_empty) * 0.3), int(len(non_empty) * 0.8)]
    return [int(non_empty.index[p]) for p in pos]


cubit_indices = pick_representative_indices(df_cubit, k=2)
dc_indices = pick_representative_indices(df_dc_train, k=2)

samples = [
    ("CUBIT-Seg", cubit_ds[cubit_indices[0]]),
    ("CUBIT-Seg", cubit_ds[cubit_indices[1]]),
    ("DeepCrack", dc_train[dc_indices[0]]),
    ("DeepCrack", dc_train[dc_indices[1]]),
]

fig, axes = plt.subplots(len(samples), 3, figsize=(11, 3.2 * len(samples)))
for row, (ds_name, sample) in enumerate(samples):
    img = sample["image"]
    mask = sample["mask"]
    dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5) * 2.0

    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"{ds_name} | {sample['sample_id']}")
    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title(f"Binary GT Mask (ratio={mask.mean()*100:.2f}%)")
    im = axes[row, 2].imshow(dist, cmap="inferno")
    axes[row, 2].set_title(f"Distance Transform Width (max={dist.max():.1f}px)")
    plt.colorbar(im, ax=axes[row, 2], fraction=0.046, pad=0.04)

    for col in range(3):
        axes[row, col].axis("off")

plt.tight_layout()
samples_path = ASSETS_DIR / "eda_samples.png"
plt.savefig(samples_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {samples_path}")

## Summary of Findings

| Dataset | Images | Empty Masks | Mean Crack Ratio (%) 평균 균열 면적 비율 | Median Crack Ratio (%) 중앙값 균열 면적 비율 | Mean Width (px) | 95th Percentile Max Width (px) |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: |
| **CUBIT-Seg** (평가용 타겟) | 5,462 | 0 | 2.002% | 1.814% | 6.224 px | 36.357 px |
| **DeepCrack** (학습용 소스) | 537 | 0 | 3.537% | 2.802% | 6.545 px | 55.176 px |

1. **Severe Class Imbalance**: Foreground crack pixels occupy only **2.00%** of the image area on average in `CUBIT-Seg` (median **1.81%**) and **3.54%** in `DeepCrack`. Evaluation must rely on **Pixel F1 (Dice), IoU, and Boundary-Tolerant F1** rather than pixel accuracy.
2. **Domain Gap (Close-up Pavement vs. UAV Building Facade)**: `DeepCrack` contains wider, higher-contrast cracks (95th percentile max width **55.2 px**), whereas `CUBIT-Seg` contains finer, thinner facade cracks (95th percentile max width **36.4 px**). Models trained on `DeepCrack` require scale/contrast normalization and careful post-processing when transferred to `CUBIT-Seg`.